# 01 - Data Acquisition

This notebook rebuilds the raw dataset from the public GeoNet APIs. Nothing is
read from a pre-prepared file, so the whole thing can be reconstructed from
scratch by anyone with an internet connection.

Two endpoints are used:

- `quakesearch.geonet.org.nz/geojson` for the earthquake catalogue
- `api.geonet.org.nz/intensity` for Felt RAPID Report submissions

Responses are cached under `data/raw`, so re-running this notebook does not hit
the API again.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import pandas as pd

import ingest

pd.set_option("display.width", 120)

## The catalogue

GeoNet publishes every located event. The window used here is magnitude 4.0 to
8.0 between October 2016 and December 2025, which is the same period the
original study covered.

In [ ]:
raw_events = ingest.fetch_events(4.0, 8.0, "2016-10-01T00:00:00", "2025-12-31T23:59:59")
print(f"{len(raw_events):,} entries returned by the catalogue")
raw_events.head()

## Two problems with the raw catalogue

The catalogue is not a list of New Zealand earthquakes. It is a list of events
New Zealand instruments detected, which is a different thing.

Querying a single month returns earthquakes in Chile and Japan. Those are real
events, but a hypocentral distance measured from Fukushima to Christchurch is
not a meaningful predictor of shaking felt here.

There are also repeated entries for the same physical earthquake, differing
only in their magnitude solution.

In [ ]:
november = ingest.fetch_events(6.0, 8.0, "2016-11-01T00:00:00", "2016-11-30T00:00:00")
november[["public_id", "origin_time", "magnitude", "longitude", "latitude"]]

Above, `2016p878059` sits at longitude -68.8, latitude -31.6, which is Chile.
The three entries ending 880764, 880765 and 880766 share an origin time and a
location off Fukushima, differing only in magnitude.

Filtering to New Zealand and collapsing duplicates removes both. The bounding
box is deliberately split across the antimeridian, because the Chatham Islands
sit just past 180 degrees and a naive 165 to 180 box would discard them.

In [ ]:
events = ingest.drop_duplicate_events(ingest.filter_to_new_zealand(raw_events))

print(f"raw catalogue        {len(raw_events):>7,}")
print(f"after NZ filter      {len(ingest.filter_to_new_zealand(raw_events)):>7,}")
print(f"after deduplication  {len(events):>7,}")
print(f"removed              {len(raw_events) - len(events):>7,} "
      f"({100 * (1 - len(events) / len(raw_events)):.0f}% of the raw catalogue)")

Almost three quarters of the raw catalogue is unusable for this problem. That
is worth knowing before trusting any count of "New Zealand earthquakes" taken
straight from the API.

## Why the selection has to be stratified

Earthquake magnitudes follow a power law: small events are overwhelmingly more
common than large ones. Sampling the catalogue at random would produce a
dataset that is almost entirely small earthquakes, and the model would rarely
see the strong shaking it most needs to predict.

In [ ]:
binned = ingest.assign_magnitude_bins(events)
counts = ingest.summarise_bins(binned)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts["magnitude_bin"], counts["events"], color="steelblue")
ax.set_yscale("log")
ax.set_xlabel("Magnitude bin")
ax.set_ylabel("Events (log scale)")
ax.set_title("New Zealand earthquakes 2016-2025, by magnitude")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

counts

Note the log scale: there are more than two thousand events in the smallest bin
and a single one above magnitude 7.5, the 2016 Kaikoura earthquake.

Sampling evenly across bins corrects for this. Bins holding fewer events than
the cap contribute everything they have, so the result is as balanced as the
catalogue allows.

## Fetching felt reports

Each event's felt reports are fetched separately. GeoNet aggregates individual
public submissions to points before publishing, and crucially each point
carries `count_mmi`: the full distribution of intensities reported there, not
just a summary. Keeping that distribution is what allows the choice of central
tendency measure to be made later rather than being fixed here.

In [ ]:
events, felt = ingest.build_dataset()

## Balancing events does not balance data

This is the finding that matters most from this notebook. Stratifying by
magnitude balances the number of *earthquakes* per bin. It does not balance the
amount of *data*, because small earthquakes generate almost no felt reports.

In [ ]:
yield_by_bin = (
    felt.groupby("public_id")["report_count"].sum()
    .to_frame("reports")
    .join(events.set_index("public_id")[["magnitude", "magnitude_bin"]])
    .groupby("magnitude_bin")
    .agg(events=("reports", "size"), median_reports=("reports", "median"),
         total_reports=("reports", "sum"))
    .astype(int)
)
yield_by_bin

A handful of large earthquakes contribute the overwhelming majority of the
reports. Any claim that this dataset is "balanced across magnitudes" is true
only in a narrow sense, and that caveat carries through to every result built
on it.

## A third data quality problem

The Felt RAPID Report survey presents six cartoons corresponding to MMI 3
through 8, so no other value should exist. A handful of MMI 1 and 2 reports do
appear in the archive. They are dropped during ingestion, with the affected
events named rather than removed silently, and `report_count` is recomputed
from the levels actually kept.

## Result

In [ ]:
print(f"events   {len(events):>8,}")
print(f"locations{len(felt):>8,}")
print(f"reports  {int(felt['report_count'].sum()):>8,}")

events.to_csv("../data/processed/events.csv", index=False)
felt.to_csv("../data/processed/felt_reports.csv", index=False)
print("\nwritten to data/processed/")

## Summary

- The raw catalogue needed heavy filtering: 72% of entries were teleseismic
  events or duplicate magnitude solutions.
- Selection is stratified across magnitude bins because the catalogue follows a
  power law, but this balances events rather than data volume.
- Felt report distributions are preserved rather than collapsed, so the target
  definition stays open.

Next: `02_exploratory_analysis.ipynb` aggregates these reports onto a grid and
examines what they actually show.